### Muntatge de Google Drive a Colab
Aquest codi connecta el  Google Drive amb el notebook de Google Colab.



In [3]:
from google.colab import drive
drive.mount('/content/drive1')

Mounted at /content/drive1


In [4]:
import os
import time
import torch
import numpy as np
import pandas as pd
import torchvision
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib import gridspec
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

### Configuració de rutes i paràmetres d’avaluació
Defineix les carpetes on hi ha els **models** i on es guardaran els **resultats**, crea la carpeta de sortida si no existeix, i configura:

- el **dispositiu** (GPU/CPU),
- el **llindar mínim de score** per filtrar deteccions,
- els **nivells de falsos positius per imatge** per calcular el NODE21 score,
- i els criteris d’**early stop** per tallar l’avaluació si el model és massa dolent.


In [5]:
MODELS_DIR = "/content/drive1/MyDrive/ML/DATASETS/NODE21/MODELS1"
OUTPUT_DIR = "/content/drive1/MyDrive/ML/DATASETS/NODE21/RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SCORE_THRESH = 0.005

FP_LEVELS = (0.25, 0.5, 1, 2, 4, 8)

EARLY_STOP = {
    "min_images": 40,
    "min_sensitivity": 0.15,
    "max_fp_per_image": 20.0,
    "min_mean_iou": 0.05
}

### Definició de rutes del dataset NODE21
Assigna les rutes principals del conjunt de dades processat:

- **PATH_IMAGES**: carpeta on hi ha les imatges en format PNG.
- **PATH_METADATA**: fitxer CSV amb les anotacions (labels i bounding boxes).

Finalment, imprimeix les rutes per comprovar que són correctes.


In [6]:
BASE_PATH = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data"

PATH_IMAGES = f"{BASE_PATH}/images"
PATH_METADATA = f"{BASE_PATH}/metadata_augmented.csv"
PATH_METADATA = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv"
#PATH_METADATA = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata.csv"

print(PATH_IMAGES)
print(PATH_METADATA)

/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/images
/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv


### Dataset NODE21 (detecció amb PNG)
Carrega imatges PNG en gris, les normalitza i les duplica a **3 canals**.  
Per a cada imatge, agrupa totes les seves **bounding boxes** del CSV i retorna `(image, target)` en format compatible amb Faster R-CNN.


In [7]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2

class Node21DetectionDatasetPNG(Dataset):
    def __init__(self, df):
        # Hacemos una copia para no modificar el df original
        self.df = df.copy()

        # Filtro: solo filas con la ruta válida
        self.df = self.df[self.df["file_path"].apply(os.path.exists)]

        # Lista de indices del df, en su orden natural (MUY IMPORTANTE)
        self.indices = self.df.index.tolist()


    def __len__(self):
        return len(self.indices)


    def __getitem__(self, idx):
        # Obtener el índice REAL en el DataFrame
        real_idx = self.indices[idx]

        # Extraer toda la fila para esa imagen
        row = self.df.loc[real_idx]
        path = row["file_path"]

        # ===============================
        # 1. Cargar imagen PNG
        # ===============================
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32)

        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        # Convertir a 3 canales
        img3 = np.stack([img, img, img], axis=0)

        # ===============================
        # 2. Obtener TODAS las anotaciones de esta imagen
        # ===============================
        rows = self.df[self.df["file_path"] == path]

        boxes = []
        labels = []

        for _, r in rows.iterrows():
            if r["label"] == 1:
                x1 = float(r["x"])
                y1 = float(r["y"])
                x2 = x1 + float(r["width"])
                y2 = y1 + float(r["height"])
                boxes.append([x1, y1, x2, y2])
                labels.append(1)

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "area": area,
            "iscrowd": iscrowd,
            "image_id": torch.tensor([real_idx])
        }

        return torch.tensor(img3, dtype=torch.float32), target

### Split + DataLoaders (detecció)
Divideix el dataset en **train/val (80/20)** i crea DataLoaders amb `collate_fn` per gestionar targets variables (bounding boxes).


In [25]:
# 1. Collate
def detection_collate(batch):
    imgs = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    return imgs, targets

# 2. Split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Datasets
train_dataset = Node21DetectionDatasetPNG(train_df)
val_dataset   = Node21DetectionDatasetPNG(val_df)

# 4. Dataloaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=detection_collate,
                          num_workers=2, pin_memory=True)

val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False,
                          collate_fn=detection_collate,
                          num_workers=2, pin_memory=True)

### Càrrega del metadata i comptatge de classes
Carrega el fitxer CSV, genera la ruta completa de cada imatge `.png` i calcula quants casos hi ha **negatius (0)** i **positius (1)**.


In [26]:
import pandas as pd
import os

df = pd.read_csv(PATH_METADATA)

df["file_path"] = df["img_name"].apply(
    lambda x: f"{PATH_IMAGES}/{x.replace('.mha', '.png')}"
)

df.head()
num_neg = (df["label"] == 0).sum()
num_pos = (df["label"] == 1).sum()
print("Negativos:", num_neg)
print("Positivos:", num_pos)

Negativos: 3748
Positivos: 4427


### Selecció del tipus de model segons el nom del checkpoint
Aquesta funció detecta si el fitxer conté la paraula **"efficientnet"** i retorna el constructor corresponent (**Faster R-CNN + EfficientNet**).  
Si no, assumeix el model estàndard (**Faster R-CNN + ResNet**).


In [27]:
def get_model_builder(checkpoint_name):
    if "efficientnet" in checkpoint_name.lower():
        return "frcnn_efficientnet"
    else:
        return "frcnn_resnet"

### Backbone EfficientNet-B0 per Faster R-CNN
Aquesta funció crea un **EfficientNet-B0** i n’extreu només la part de **features** (backbone).  
S’indica que la seva sortida té **1280 canals**, per poder-la connectar correctament a Faster R-CNN.


In [12]:
from torchvision.models import efficientnet_b0

def build_my_efficientnet_backbone():
    model = efficientnet_b0(weights=None)
    backbone = model.features
    backbone.out_channels = 1280
    return backbone

### Faster R-CNN amb backbone EfficientNet (Node21)
Aquesta funció construeix un **Faster R-CNN** utilitzant un backbone **EfficientNet**.  
Defineix un **AnchorGenerator** simple (1 mida i 3 proporcions) i configura el detector per **2 classes** (fons + nòdul).


In [13]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator

def build_frcnn_efficientnet(backbone):
    backbone.out_channels = 1280

    anchor_generator = AnchorGenerator(
        sizes=((128,),),              # 1 escala
        aspect_ratios=((0.5, 1.0, 2.0),)  # 3 ratios
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=2,
        rpn_anchor_generator=anchor_generator
    )
    return model


### Faster R-CNN amb ResNet50 + FPN (Node21)
Aquesta funció crea un **Faster R-CNN ResNet50-FPN** des de zero (sense pesos preentrenats)  
i substitueix el capçal de classificació per treballar amb **2 classes** (fons + nòdul).


In [14]:
def build_frcnn_resnet():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    return model

### Constructor genèric de Faster R-CNN (amb nombre de classes configurable)
Aquesta funció crea un **Faster R-CNN ResNet50-FPN** des de zero i adapta el *head* final perquè pugui detectar el nombre de classes indicat (per defecte **2: fons + nòdul**).


In [15]:
def build_model(num_classes=2):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, num_classes
    )

    return model

### Càrrega segura d’un checkpoint (compatible o descartat)
Aquesta funció intenta carregar un *checkpoint* dins del model.  
Si falten pesos o en sobren (model incompatible) o hi ha un error, retorna **False** i mostra el motiu.  
Si tot encaixa correctament, retorna **True**.


In [16]:
def load_checkpoint_safe(model, path, device):
    try:
        state = torch.load(path, map_location=device)
        missing, unexpected = model.load_state_dict(state, strict=False)

        if missing or unexpected:
            print("   Incompatible")
            return False

        return True
    except Exception as e:
        print("   Error:", e)
        return False

### Càlcul de la IoU (Intersection over Union)
Aquesta funció calcula la **IoU** entre dues *bounding boxes*.  
Mesura el grau de solapament entre elles com:

**IoU = àrea d’intersecció / àrea d’unió**.

Retorna un valor entre **0 i 1** (0 = no se solapen, 1 = coincideixen totalment).


In [17]:
def compute_iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])

    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

### Recollida de deteccions amb *early stopping*
Aquesta funció recorre el *dataset*, fa inferència amb el model i guarda les deteccions (*boxes* i *scores*) junt amb les GT.  
A mesura que avança, calcula mètriques ràpides (**sensibilitat**, **FP/imatge** i **IoU mitjana**) i **atura l’avaluació abans d’acabar** si el model és clarament dolent, estalviant temps.


In [18]:
def collect_detections_early_stop(
    model, dataset, device, score_thresh, early_cfg
):
    model.eval()

    detections = []
    gt_count = tp = fp = iou_sum = iou_n = 0
    n_images = len(dataset)

    with torch.no_grad():
        for i in range(n_images):
            img, target = dataset[i]

            # IMAGEN A GPU
            img = img.to(device, non_blocking=True)

            gt_boxes = target["boxes"].cpu().numpy()
            gt_count += len(gt_boxes)

            # INFERENCIA EN CUDA
            pred = model([img])[0]

            boxes = pred["boxes"].detach().cpu().numpy()
            scores = pred["scores"].detach().cpu().numpy()

            keep = scores >= score_thresh
            boxes = boxes[keep]
            scores = scores[keep]

            # matching CPU
            matched = set()
            for pb in boxes:
                best_iou, best_gt = 0.0, -1
                for j, gt in enumerate(gt_boxes):
                    if j in matched:
                        continue
                    iou = compute_iou(pb, gt)
                    if iou > best_iou:
                        best_iou, best_gt = iou, j

                if best_iou >= 0.2:
                    tp += 1
                    matched.add(best_gt)
                    iou_sum += best_iou
                    iou_n += 1
                else:
                    fp += 1

            detections.append({
                "gt_boxes": gt_boxes,
                "pred_boxes": boxes,
                "scores": scores
            })


            if i % 50 == 0 or i == n_images - 1:
                print(
                    f"    Inferencia: {i+1}/{n_images} imágenes",
                    end="\r"
                )

            # -------- EARLY STOP --------
            if i + 1 >= early_cfg["min_images"]:
                sens = tp / (gt_count + 1e-8)
                fp_img = fp / (i + 1)
                mean_iou = iou_sum / iou_n if iou_n else 0.0

                if (
                    sens < early_cfg["min_sensitivity"]
                    or fp_img > early_cfg["max_fp_per_image"]
                    or mean_iou < early_cfg["min_mean_iou"]
                ):
                    print(
                        f"\nEarly stop @ {i+1} | "
                        f"Sens={sens:.3f} | "
                        f"FP/img={fp_img:.1f} | "
                        f"IoU={mean_iou:.3f}"
                    )
                    return None, None

    print()
    return detections, gt_count




### Càlcul de la corba FROC (FP/imatge vs sensibilitat)
Aquesta funció construeix la **corba FROC** a partir de les deteccions del model.  
Per cada imatge ordena les prediccions per *score*, les compara amb les GT amb un llindar d’**IoU**, i marca cada predicció com a **TP** o **FP** (evitant reutilitzar la mateixa GT).  
Finalment acumula TP/FP globalment i retorna:

- **fps** = falsos positius per imatge  
- **sens** = sensibilitat (TP / total GT)


In [19]:
def compute_froc(detections, iou_thresh=0.2):
    all_tp, all_fp, all_scores = [], [], []
    total_gt = sum(len(d["gt_boxes"]) for d in detections)
    num_images = len(detections)

    for det in detections:
        matched = set()
        boxes, scores = det["pred_boxes"], det["scores"]

        order = np.argsort(scores)[::-1]
        boxes, scores = boxes[order], scores[order]

        for i, pb in enumerate(boxes):
            best_iou, best_gt = 0.0, -1
            for j, gt in enumerate(det["gt_boxes"]):
                if j in matched:
                    continue
                iou = compute_iou(pb, gt)
                if iou > best_iou:
                    best_iou, best_gt = iou, j

            if best_iou >= iou_thresh:
                all_tp.append(1)
                all_fp.append(0)
                matched.add(best_gt)
            else:
                all_tp.append(0)
                all_fp.append(1)

            all_scores.append(scores[i])

    order = np.argsort(all_scores)[::-1]
    tp = np.cumsum(np.array(all_tp)[order])
    fp = np.cumsum(np.array(all_fp)[order])

    sens = tp / (total_gt + 1e-8)
    fps = fp / num_images

    return fps, sens


### Avaluació d’un model amb mètrica NODE21 i corba FROC
Aquesta funció avalua un detector sobre un *dataset* i retorna un resum de rendiment.

- Genera deteccions amb `collect_detections_early_stop()` (si el model és dolent, pot parar abans).
- Calcula la **corba FROC** (`fps`, `sens`) amb `compute_froc()`.
- Obté la **millor sensibilitat** per a cada nivell de falsos positius per imatge (`FP_LEVELS`).
- Calcula el **NODE21 score** com la mitjana d’aquestes sensibilitats.
- Dibuixa la corba FROC i marca els punts de sensibilitat per cada FP.
- Retorna el `node21_score`, les sensibilitats i la figura generada.


In [20]:
def evaluate_model(model, dataset, device, name):
    model.eval()
    detections, gt_count = collect_detections_early_stop(
        model, dataset, device, SCORE_THRESH, EARLY_STOP
    )

    if detections is None:
        return {"early_stopped": True}

    fps, sens = compute_froc(detections)

    sens_at_fp = {
        fp: np.max(sens[fps <= fp]) if np.any(fps <= fp) else 0.0
        for fp in FP_LEVELS
    }

    node21 = float(np.mean(list(sens_at_fp.values())))

    # -------- Plot --------
    fig = plt.figure(figsize=(8, 6))
    plt.plot(fps, sens, lw=2)
    for fp, s in sens_at_fp.items():
        plt.scatter(fp, s)
        plt.text(fp, s, f"{s:.2f}", fontsize=9)

    plt.xscale("log")
    plt.xlabel("FP / image")
    plt.ylabel("Sensitivity")
    plt.title(name)
    plt.grid(True, which="both")
    plt.tight_layout()

    return {
        "node21_score": node21,
        "sensitivities": sens_at_fp,
        "figure": fig
    }


In [21]:
from torchvision.models.detection import retinanet_resnet50_fpn

### Construcció d’un model RetinaNet (ResNet50 + FPN)
Aquesta funció crea un detector **RetinaNet** amb backbone **ResNet50+FPN**.

- No carrega pesos preentrenats (`weights=None`, `weights_backbone=None`).
- Configura el detector per a **2 classes** (fons + nòdul).
- Retorna el model llest per entrenar o carregar un checkpoint.


In [22]:
def build_retinanet():
    model = retinanet_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=2
    )
    return model


### Inferència automàtica del backbone des d’un checkpoint
Aquesta funció llegeix un *checkpoint* i intenta deduir quin **backbone** s’ha utilitzat.

- Carrega el `state_dict` del model.
- Cerca el pes de la capa **RPN conv** (`rpn.head.conv...weight`).
- Si detecta **1280 canals**, assumeix **Faster R-CNN + EfficientNet**.
- Si detecta **256 canals**, assumeix **Faster R-CNN + ResNet50-FPN**.
- Si no pot deduir-ho, llança un error.


In [23]:
def infer_builder_from_checkpoint(path, device):
    state = torch.load(path, map_location=device)
    # RPN conv: rpn.head.conv.0.0.weight
    for k, v in state.items():
        if "rpn.head.conv" in k and "weight" in k:
            in_ch = v.shape[0]
            if in_ch == 1280:
                return "frcnn_efficientnet"
            elif in_ch == 256:
                return "frcnn_resnet"
    raise ValueError("No se puede inferir el backbone del checkpoint")


### Bucle d’avaluació massiva de models (RetinaNet)
Aquest codi recorre tots els fitxers `.pth` d’un directori i **avalua cada model automàticament**.

Per a cada checkpoint:
- Construeix un **RetinaNet nou** (`build_retinanet()`).
- Intenta carregar els pesos amb `load_checkpoint_safe()` (si no encaixa, es descarta).
- Avalua el model amb `evaluate_model()` (FROC + NODE21 score).
- Si passa l’early-stop, guarda la gràfica **FROC** en PNG.
- Desa els resultats (`node21_score`, temps i sensibilitats per FP/img) dins `results`.
- Els models descartats s’afegeixen a `skipped`.

Al final mostra el **temps total** dedicat a l’avaluació completa.


In [ ]:
results = []
skipped = []

model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith(".pth")]

start_all = time.time()

for fname in tqdm(model_files, desc="Evaluando modelos"):
    print(f"\n {fname}")

    # decidir qué arquitectura usar
    ckpt_path = os.path.join(MODELS_DIR, fname)
    #builder = infer_builder_from_checkpoint(ckpt_path, DEVICE)

    model = build_retinanet().to(DEVICE)

    # cargar checkpoint
    if not load_checkpoint_safe(
        model,
        os.path.join(MODELS_DIR, fname),
        DEVICE
    ):
        skipped.append(fname)
        continue

    t0 = time.time()
    out = evaluate_model(model, val_dataset, DEVICE, fname)

    if out.get("early_stopped"):
        skipped.append(fname)
        continue

    out["figure"].savefig(
        os.path.join(OUTPUT_DIR, fname.replace(".pth", "_froc.png")),
        dpi=300
    )

    results.append({
        "model": fname,
        "node21_score": out["node21_score"],
        "time_sec": round(time.time() - t0, 1),
        **{f"sens@{fp}": s for fp, s in out["sensitivities"].items()}
    })


    print("Resultados:")
    print(f"   Model: {fname}")
    print(f"   NODE21 score: {out['node21_score']:.4f}")
    print(f"   Tiempo: {round(time.time() - t0, 1)} s")

    for fp, s in out["sensitivities"].items():
        print(f"   Sens@{fp} FP/img: {s:.3f}")



print(f"\nTotal tiempo: {(time.time()-start_all)/60:.1f} min")


### Resum final i exportació de resultats
Aquest bloc converteix la llista `results` en un *DataFrame*, l’ordena pel **NODE21 score** (de millor a pitjor) i el guarda en un CSV.

Després:
- Mostra per pantalla el **millor model** (`df.iloc[0]`).
- Llista els **models descartats** durant el procés (`skipped`).


In [29]:
df = pd.DataFrame(results).sort_values("node21_score", ascending=False)
df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_retinanet.csv"), index=False)

print("\n Mejor modelo:")
print(df.iloc[0])

print("\n Modelos descartados:")
for m in skipped:
    print(" -", m)



 Mejor modelo:
model           checkpoint_retinanet_epoch_04_20260102_110739.pth
node21_score                                             0.741592
time_sec                                                    180.8
sens@0.25                                                 0.46953
sens@0.5                                                 0.649351
sens@1                                                   0.778222
sens@2                                                   0.825175
sens@4                                                   0.854146
sens@8                                                   0.873127
Name: 4, dtype: object

 Modelos descartados:


In [ ]:
os.listdir("/content/drive1/MyDrive/ML/DATASETS/NODE21")

['proccessed_data',
 'augmented_pos',
 'metadata_balanced.csv',
 'efficientnet_b0_multicanal.pth',
 'efficientnet_b0_multicanal_laplaciana.pth',
 'efficientnet_b0_multicanal_Unsharp.pth',
 'efficientnet_b0_multicanal_Unsharp_bo.pth',
 'original_data',
 'original_augmented_pos',
 'metadata_original_balanced.csv',
 'best_frcnn_node21_1024.pth',
 'best_frcnn_node21.pth',
 'metadata_balanced.gsheet',
 'models',
 'MODELS1',
 'RESULTS']